# CoREMOF-tools: CR/NCR classification and leakage-safe splitting

This executable notebook is the companion to the [dataset-splitting handbook](../README_DATASET_SPLITTING.md). It shows how to:

1. locate and validate an extracted CoRE-MOF release;
2. inspect current public structure metadata;
3. recompute 3-, 4-, and 5-checker consensus labels;
4. filter structures for a modelling study;
5. inspect parent-group evidence;
6. make a deterministic train/validation/test split without crossing known leakage blocks; and
7. save and verify the assignment CSV and reproducibility receipt.

The splitter reads completed release metadata. It does **not** run the five checkers, calculate RACs/MOFids/Zeo++, edit CIFs, or impute unavailable results. Current v26 inputs are provisional audited snapshots, so the resulting splits are exploratory and are not official CoRE-MOF benchmark assignments.

## 1. Installation

Run one of the following commands in a terminal, or remove the leading `#` from the appropriate line below. The validated development wheel is used on the project machine because the public PyPI release may not yet contain this API.

In [ ]:
# Install an audited wheel supplied by the project:
# %pip install /path/to/coremof_tools-0.4.0.dev0-py3-none-any.whl

# Editable source installation for developers:
# %pip install -e /path/to/CoRE-MOF-Tools

## 2. Imports and portable path configuration

For use on another machine, set `COREMOF_RELEASE` (or the release-specific `COREMOF_V2602_RELEASE`) to the extracted directory containing `dataset_info.json`. Optionally set `COREMOF_NOTEBOOK_OUTPUT` to choose where split files are written.

In [ ]:
import csv
import json
import os
from collections import Counter
from pathlib import Path

from CoREMOF import __version__
from CoREMOF.dataset import CoREMOFDataset
from CoREMOF.labels import CHECKER_COLUMNS, CHECKER_PRESETS
from CoREMOF.parents import ParentResolver

print("CoREMOF-tools version:", __version__)

In [ ]:
def find_release_root():
    candidates = []
    configured = os.environ.get("COREMOF_RELEASE") or os.environ.get("COREMOF_V2602_RELEASE")
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend([
        Path.cwd() / "coremof_v26.0.2",
        Path.cwd().parent / "coremof_v26.0.2",
    ])
    for candidate in candidates:
        if (candidate / "dataset_info.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "No extracted CoRE-MOF release was found. Set COREMOF_RELEASE "
        "to the directory containing dataset_info.json."
    )

RELEASE_ROOT = find_release_root()
OUTPUT_ROOT = Path(
    os.environ.get("COREMOF_NOTEBOOK_OUTPUT", Path.cwd() / "coremof_notebook_outputs")
).expanduser().resolve()
RUN_STRICT_CIF_CHECK = False

print("Release root:", RELEASE_ROOT)
print("Output root: ", OUTPUT_ROOT)

## 3. Load and validate the release

Normal loading validates the release tables, declared manifest entries, IDs, labels, and parent-group contract. It does not read and hash every CIF byte. That more expensive check is available separately below.

In [ ]:
dataset = CoREMOFDataset.from_release(
    RELEASE_ROOT, verify_cif_files=RUN_STRICT_CIF_CHECK
)

print("Dataset version:    ", dataset.dataset_version)
print("Structures:         ", f"{len(dataset):,}")
print("Release status:     ", dataset.dataset_info.get("release_status"))
print("Parent input status:", dataset.parent_group_methods.get("release_status"))
print("CIF bytes verified: ", dataset.cif_files_verified)
print("Input tables hashed: ", len(dataset.input_hashes))

### Optional strict CIF-byte validation

Set `RUN_STRICT_CIF_CHECK = True` in the configuration cell above when you want the primary dataset object to rehash every CIF and compare it with `manifests/cif_manifest.csv`. This is recommended once for a newly copied or extracted release, but is disabled by default in the tutorial because it reads the complete CIF corpus. The verification state is carried into the split receipt.

In [ ]:
if RUN_STRICT_CIF_CHECK:
    assert dataset.cif_files_verified
    print(f"Verified all {len(dataset):,} CIF files in the primary dataset object.")
else:
    print("Skipped strict CIF-byte validation.")

## 4. Inspect one structure

A `StructureRecord` joins the main metadata row, criterion-specific parent memberships, and the CIF manifest entry. Values read from the public metadata CSV are strings. The five checker statuses are votes or explicit non-votes; they are not error-severity scores.

In [ ]:
record = dataset[0]
checker_statuses = {
    checker: record.metadata[column]
    for checker, column in CHECKER_COLUMNS.items()
}
parent_preview = {}
for method in ("rac5_zeo", "rac5", "zeo", "source_id", "mofid_v2", "mofid_v1"):
    group = record.parent_group(method)
    parent_preview[method] = {
        "status": group.status,
        "group_id": group.group_id,
        "size": group.size,
    }

structure_summary = {
    "structure_id": record.structure_id,
    "source_database": record.source_database,
    "source_id": record.get("source_id"),
    "structure_variant": record.structure_variant,
    "metal_elements": record.metal_elements,
    "doi": record.get("doi"),
    "mofid_v1": record.get("mofid_v1"),
    "mofid_v2": record.get("mofid_v2"),
    "checker_statuses": checker_statuses,
    "parent_groups": parent_preview,
    "cif_manifest": dict(record.cif_manifest) if record.cif_manifest else None,
}
print(json.dumps(structure_summary, indent=2))

## 5. Recompute CR/NCR consensus

For every selected checker set, the rule is strict:

- all included checkers `PASS` → `CR`;
- all included checkers `FAIL` → `NCR`;
- a complete mixture of `PASS` and `FAIL` → `AMBIGUOUS`;
- any unavailable/error/timeout non-vote → `UNCHECKED`.

An execution error is never converted into `FAIL` or `NCR`.

In [ ]:
classified_views = {}
label_order = ("CR", "NCR", "AMBIGUOUS", "UNCHECKED")

print("view       checkers  CR       NCR      AMBIGUOUS  UNCHECKED")
for view_name in ("3checker", "4checker", "5checker"):
    view = dataset.classify(view_name)
    classified_views[view_name] = view
    counts = view.label_counts()
    values = [counts.get(label, 0) for label in label_order]
    print(f"{view_name:<11}{len(CHECKER_PRESETS[view_name]):<10}" + "".join(f"{value:<10}" for value in values))
    assert sum(values) == len(dataset)

five_checker = classified_views["5checker"]

### Optional user-defined checker experiment

Python accepts an explicit ordered checker sequence. Such a result is marked `USER_DEFINED` and must not be described as an official release checker view.

In [ ]:
custom_view = dataset.classify(("MOFClassifier", "Chen-Manz"))
print("Identifier:    ", custom_view.checker_view)
print("Official view: ", custom_view.checker_view_official)
print("Label counts:  ", dict(custom_view.label_counts()))
assert not custom_view.checker_view_official

## 6. Filter a modelling cohort

The following example keeps classified CR/NCR structures from the COD or SI sources, uses ASR/FSR variants, and requires Cu or Zn. Different filter categories are combined with AND; Cu/Zn are combined with OR. Filtering does not erase full-release parent bridges used by the recommended leakage guard.

In [ ]:
modelling_subset = five_checker.filter(
    labels=("CR", "NCR"),
    sources=("COD", "SI"),
    variants=("ASR", "FSR"),
    metals=("Cu", "Zn"),
)

source_counts = Counter(record.source_database for record in modelling_subset)
print("Selected structures:", f"{len(modelling_subset):,}")
print("Labels:             ", dict(modelling_subset.label_counts()))
print("Sources:            ", dict(sorted(source_counts.items())))
print("First five IDs:     ", modelling_subset.structure_ids[:5])

## 7. Inspect the recommended parent hierarchy

`priority_main` is the conflict-aware hierarchy RAC5 → MOFid v2 → MOFid v1. It is not a simple row-wise fallback. Resolution is constructed over the complete release before the requested preview is returned. Missing evidence becomes a unique singleton by default, so two null values never match.

In [ ]:
resolver = ParentResolver(dataset)
preview_ids = modelling_subset.structure_ids[:12]
parent_resolution = resolver.resolve(
    "priority_main", structure_ids=preview_ids
)

for structure_id in preview_ids:
    print(
        structure_id,
        parent_resolution.groups.get(structure_id),
        parent_resolution.evidence_by_id.get(structure_id),
        parent_resolution.exclusions.get(structure_id),
    )
print("Relevant conflict groups in preview:", len(parent_resolution.conflicts))

## 8. Build the recommended leakage-safe split

This section intentionally returns to the broader full `five_checker` view and applies only the COD/SI and CR/NCR filters; it does not reuse the Cu/Zn and ASR/FSR restrictions in `modelling_subset`. To split that exact smaller cohort instead, call `modelling_subset.train_valid_test_split(...)`; its preselection will be recorded in the receipt.

Here `leakage_guard="auto"` resolves to `main_union` for `priority_main`. The indivisible components are built on the full COD+CSD+SI universe from full CIF SHA-256, database-namespaced source siblings, and available release-authorized RAC5, MOFid v2, and MOFid v1 groups before experiment filters are applied. This recommended guard requires `manifests/cif_manifest.csv` with one full SHA-256 value for every release structure.

In [ ]:
split = five_checker.train_valid_test_split(
    parent_method="priority_main",
    leakage_guard="auto",
    labels=("CR", "NCR"),
    sources=("COD", "SI"),
    fractions=(0.8, 0.1, 0.1),
    random_state=42,
    missing_parent="singleton",
    stratify_by=("label",),
)

print("Counts:            ", dict(split.counts))
print("Achieved fractions:", dict(split.achieved_fractions))
print("Labels by split:   ", {k: dict(v) for k, v in split.label_counts_by_split.items()})
print("Warnings:          ", split.warnings)

In [ ]:
audit = split.leakage_audit
assert audit["passed"]
assert audit["cross_split_block_count"] == 0
assert set(split.train_ids).isdisjoint(split.validation_ids)
assert set(split.train_ids).isdisjoint(split.test_ids)
assert set(split.validation_ids).isdisjoint(split.test_ids)

print("Resolved leakage guard:", split.leakage_guard)
print("Leakage blocks:       ", audit["block_count"])
print("Largest block:        ", audit["max_block_size"])
print("Crossed blocks:       ", audit["cross_split_block_count"])
print("Provisional input:    ", split.provisional_input)
print("Official split:       ", split.official_split)

## 9. Inspect and save the reproducibility receipt

Persist structure IDs rather than positional indices. The CSV contains every release row, including explicit exclusions. The JSON binds inputs, parameters, source-code hashes, assignments, warnings, parent conflicts, and the zero-leakage audit. The example uses `overwrite=True` so the tutorial is rerunnable; omit it in production when accidental replacement should fail closed.

In [ ]:
receipt = split.receipt()
receipt_summary = {
    "dataset_version": receipt["dataset_version"],
    "checker_view": receipt["checker_view"],
    "checker_view_kind": receipt["checker_view_kind"],
    "parent_method": receipt["parent_method"],
    "leakage_guard": receipt["leakage_guard"],
    "assignment_sha256": receipt["assignment_sha256"],
    "parent_conflict_count": receipt["parent_conflict_count"],
    "provisional_input": receipt["provisional_input"],
    "official_split": receipt["official_split"],
}
print(json.dumps(receipt_summary, indent=2))

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
csv_path, json_path = split.write(
    OUTPUT_ROOT,
    stem="cod_si_5checker_seed42",
    overwrite=True,
)

with csv_path.open(newline="", encoding="utf-8") as handle:
    assignment_rows = list(csv.DictReader(handle))
with json_path.open(encoding="utf-8") as handle:
    saved_receipt = json.load(handle)

assert len(assignment_rows) == len(dataset)
assert saved_receipt["assignment_sha256"] == receipt["assignment_sha256"]
assert saved_receipt["leakage_audit"]["passed"]

print("Assignment CSV:", csv_path)
print("Receipt JSON: ", json_path)
print("First assignment row:")
print(json.dumps(assignment_rows[0], indent=2))

## 10. Join split IDs to metadata and per-structure JSON

Use the stable IDs in `split.train_ids`, `split.validation_ids`, and `split.test_ids` to join targets with descriptors or CIFs. The per-structure JSON records provide the current checker detail and available RAC5, Zeo++, topology, and other release properties.

In [ ]:
train_preview = []
for structure_id in split.train_ids[:5]:
    item = dataset[structure_id]
    train_preview.append({
        "structure_id": structure_id,
        "target": split.labels[structure_id],
        "source_database": item.source_database,
        "structure_variant": item.structure_variant,
        "metal_elements": item.metal_elements,
        "cif_file": item.get("cif_file"),
    })
print(json.dumps(train_preview, indent=2))

example_id = split.train_ids[0]
structure_json_path = RELEASE_ROOT / "metadata" / "structures" / f"{example_id}.json"
if structure_json_path.is_file():
    with structure_json_path.open(encoding="utf-8") as handle:
        structure_json = json.load(handle)
    print("Per-structure JSON sections:", sorted(structure_json))
    print("Feature sections:           ", sorted(structure_json.get("features", {})))
else:
    print("This release does not include per-structure JSON files:", structure_json_path)

## 11. Optional determinism replay

The same inputs and parameters must reproduce the same `structure_id → partition` mapping. Enable this cell when auditing an environment or copied release.

In [ ]:
RUN_DETERMINISM_REPLAY = False

if RUN_DETERMINISM_REPLAY:
    repeated = five_checker.train_valid_test_split(
        parent_method="priority_main",
        leakage_guard="auto",
        labels=("CR", "NCR"),
        sources=("COD", "SI"),
        fractions=(0.8, 0.1, 0.1),
        random_state=42,
        missing_parent="singleton",
        stratify_by=("label",),
    )
    assert dict(repeated.assignments) == dict(split.assignments)
    assert repeated.receipt()["assignment_sha256"] == receipt["assignment_sha256"]
    print("Determinism replay passed.")
else:
    print("Skipped the optional second full split.")

## 12. Equivalent CLI command

The recommended split can also be generated without Python code:

```bash
coremof split /path/to/coremof_v26.0.2 \
  --checkers 5checker \
  --parent-method priority_main \
  --leakage-guard auto \
  --labels CR NCR \
  --sources COD SI \
  --fractions 0.8 0.1 0.1 \
  --random-state 42 \
  --output-directory model_splits \
  --stem cod_si_5checker_seed42
```

Use `--verify-cifs` for the expensive strict byte-level check. Do not pass `--official`: current releases do not yet provide an audited official assignment manifest, and the package intentionally fails closed.

## Interpretation and redistribution reminders

- Treat `AMBIGUOUS` and `UNCHECKED` as distinct from NCR.
- Treat parent groups as criterion-dependent screening relations, not automatically as proof of identical frameworks.
- Keep the complete receipt with any published model or benchmark result.
- COD is the default open-data base. SI redistribution still requires asset-level rights review. CSD CIFs and structure-resolved CSD-derived data remain licence-gated unless CCDC grants permission for the intended redistribution.
- A source filter is not a licence-sanitization step: the standard assignment CSV deliberately contains all release rows, including excluded CSD rows. Prepare a separately audited public projection before redistribution.
- A future official v26.0.2 split must preserve frozen base assignments rather than independently reshuffling v26.0.1.

For the complete API, options, schemas, and troubleshooting guide, read the [dataset-splitting handbook](../README_DATASET_SPLITTING.md).